# Вариант 9
## Разработка системы интеллектуального управления пропускной способностью спутникового канала связи на основе прогнозирования трафика

**Авторская постановка проекта:** прогноз сетевой нагрузки используется как вход для rule-based управления выделяемой ёмкостью канала.


## 1. Постановка задачи

**Цель работы** — разработать имитационную систему, которая прогнозирует будущую загрузку канала и на основе прогноза заранее меняет выделяемую пропускную способность.

**Основная идея проекта:**
- прогноз трафика не является конечной целью;
- прогноз используется как управляющий сигнал;
- качество решения оценивается не только по RMSE, но и по снижению перегрузок канала.


## 2. Описание данных

В работе используется открытый датасет `CESNET-TimeSeries24 sample` из каталога `datasets/`.

Важно: это **не прямой спутниковый датасет**. Он применяется как открытый источник временного ряда нагрузки, на основе которого строится **имитационная модель спутникового канала** с ограниченной пропускной способностью.

Для эксперимента используется масштаб `1 hour`, а нагрузка канала интерпретируется через суммарный показатель `n_bytes`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from satellite_capacity_project import (
    BASE_DIR,
    build_channel_dataset,
    build_control_policy,
    create_features,
    ensure_directories,
    train_models,
    train_test_split_time,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ensure_directories()
BASE_DIR

In [ ]:
channel_df, selected_ids = build_channel_dataset(scale='1_hour', top_k=5)

print('Выбранные ряды:', ', '.join(selected_ids))
print('Размер агрегированного ряда:', channel_df.shape)

channel_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(channel_df['timestamp'], channel_df['total_n_bytes'], linewidth=1.5)
ax.set_title('Агрегированная нагрузка канала по времени')
ax.set_xlabel('Время')
ax.set_ylabel('Нагрузка, bytes/hour')
plt.show()

## 3. Подготовка признаков

Для прогноза строятся:
- календарные признаки;
- циклические признаки `sin/cos`;
- лаги `1, 2, 3, 6, 12, 24, 48, 72`;
- скользящие средние и стандартные отклонения.

**Защита от leakage:** все rolling-признаки и производные показатели рассчитываются только по прошлым значениям, со сдвигом минимум на один шаг назад.


In [ ]:
featured_df = create_features(channel_df)
train_df, test_df = train_test_split_time(featured_df)

print('Размер после feature engineering:', featured_df.shape)
print('Размер train:', train_df.shape)
print('Размер test:', test_df.shape)

featured_df[['timestamp', 'total_n_bytes', 'lag_1', 'lag_24', 'rolling_mean_24']].head()

## 4. Прогнозирование нагрузки

Сравниваются три модели:
- `NaiveLastValue`;
- `SeasonalNaive24`;
- `OLSRegression`.

Для учебной постановки этого достаточно: важно не максимизировать сложность модели, а честно получить работающий прогнозный сигнал для контура управления.


In [ ]:
model_metrics, predictions, fitted_models, feature_cols = train_models(train_df, test_df)
best_model_name = model_metrics.iloc[0]['model']
best_prediction = predictions[best_model_name]

print('Лучшая модель:', best_model_name)
model_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(model_metrics['model'], model_metrics['RMSE'], color=['steelblue', 'seagreen', 'darkorange'])
ax.set_title('Сравнение моделей по RMSE')
ax.set_ylabel('RMSE')
plt.show()

На выбранном ряде лучшей оказалась модель `NaiveLastValue`. Это объясняется высокой инерционностью трафика: предыдущее значение уже хорошо описывает ближайшее будущее. В данной работе это интерпретируется как **честный и полезный результат**, а не как слабость модели.


## 5. Интеллектуальное управление пропускной способностью

Сравниваются две политики:
- `BaselineFixedCapacity` — фиксированная ёмкость канала;
- `ForecastDrivenCapacity` — адаптивная ёмкость на основе прогноза с резервным запасом.

Ключевые KPI управления:
- `overload_rate`;
- `served_traffic_ratio`;
- `total_overload_bytes`;
- `unused capacity`.


In [ ]:
control_df, policy_metrics = build_control_policy(train_df, test_df, best_prediction)
policy_metrics

In [ ]:
plot_df = control_df.tail(240).copy()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(plot_df['timestamp'], plot_df['total_n_bytes'], label='Фактическая нагрузка', linewidth=2)
ax.plot(plot_df['timestamp'], plot_df['forecast_n_bytes'], label='Прогноз', linewidth=2, linestyle='--')
ax.plot(plot_df['timestamp'], plot_df['baseline_capacity'], label='Фиксированная ёмкость', linewidth=1.5)
ax.plot(plot_df['timestamp'], plot_df['dynamic_capacity'], label='Адаптивная ёмкость', linewidth=1.5)
ax.set_title('Прогноз нагрузки и управление пропускной способностью')
ax.set_xlabel('Время')
ax.set_ylabel('bytes/hour')
ax.legend()
plt.show()

In [ ]:
baseline = policy_metrics.loc[policy_metrics['policy'] == 'BaselineFixedCapacity'].iloc[0]
adaptive = policy_metrics.loc[policy_metrics['policy'] == 'ForecastDrivenCapacity'].iloc[0]

overload_drop = baseline['overload_rate'] - adaptive['overload_rate']
served_gain = adaptive['served_traffic_ratio'] - baseline['served_traffic_ratio']
overload_ratio = baseline['total_overload_bytes'] / adaptive['total_overload_bytes']

print(f"Снижение overload_rate: {overload_drop:.2f} п.п.")
print(f"Рост served_traffic_ratio: {served_gain:.2f} п.п.")
print(f"Сокращение total_overload_bytes: примерно в {overload_ratio:.2f} раза")

## 6. Итоговые выводы

1. На основе открытого временного ряда была построена имитационная модель спутникового канала с ограниченной пропускной способностью.
2. Лучшая прогнозная модель на выбранных данных — `NaiveLastValue`, что соответствует инерционному характеру нагрузки.
3. Adaptive policy на основе прогноза заметно снижает перегрузки по сравнению с фиксированной ёмкостью.
4. Основная интеллектуальность системы состоит в использовании прогноза как входа для модуля управления ресурсом.


## 7. Ограничения и перспективы

- датасет не является прямым спутниковым измерением;
- управление реализовано как rule-based policy;
- не учитываются физические параметры канала, например SNR и BER.

Перспективы развития:
- подключение реальных спутниковых телеметрических данных;
- добавление более сложных моделей прогноза;
- учёт задержки, джиттера и качества радиоканала в политике управления.
